# 7.5 · 实时风控与监控

> 风控是实盘的"生命线"。真金白银运行时，系统必须在下单前、运行中、收盘后持续把关，异常时自动熔断并报警。
> 本节理解实时风控的三道防线、限额体系、熔断与报警、监控与容灾。

## 一、实时风控的三道防线

| 阶段 | 时机 | 检查内容 | 动作 |
|---|---|---|---|
| **事前（pre-trade）** | 下单前 | 单票权重、现金是否够、是否涨跌停、合规、额度 | 不通过则拒绝下单 |
| **事中（intraday）** | 持仓运行中 | 实时回撤、现金比例、敞口、集中度、风险因子 | 触发阈值则报警/熔断/减仓 |
| **事后（post-trade）** | 收盘后 | 持仓、净值、限额执行情况、对账 | 留档、复盘、报告 |

> 事前防"错单"，事中防"失控"，事后做"复盘与对账"。代码在 live_lab.risk。

## 二、限额体系与事前风控

In [1]:
import sys
sys.path.insert(0, r'C:\Users\gucao\Desktop\Qlib')
from live_lab.risk import RiskManager
rm = RiskManager({'max_weight': 0.1, 'min_cash': 0.05, 'max_drawdown': 0.15})
# 事前：合规单通过，超限单被拒
for code, w in [('A', 0.05), ('B', 0.20)]:
    ok, msg = rm.pre_trade_check(code, w)
    print('%s 权重 %.2f -> %s' % (code, w, msg))

A 权重 0.05 -> 通过
B 权重 0.20 -> 拒绝：B 权重 0.20 超上限


## 三、事中风控：熔断与 kill switch

In [2]:
# 模拟盘中：现金/回撤变化，触发熔断
scenarios = [('正常', 0.10, 0.05), ('现金不足', 0.02, 0.05),
             ('回撤超限', 0.10, 0.20)]
for name, cash_r, dd in scenarios:
    ok, msg = rm.intraday_check(cash_r, dd)
    print('%s -> %s' % (name, msg))
print()
print('累计报警:', rm.alerts)
print()
print('kill switch：触发严重熔断时，一键停止开新仓、只减不增，必要时全部平仓并通知人工。')

正常 -> 正常
现金不足 -> 熔断：现金比例 0.02 过低
回撤超限 -> 熔断：回撤 0.20 超阈值

累计报警: ['拒绝：B 权重 0.20 超上限', '熔断：现金比例 0.02 过低', '熔断：回撤 0.20 超阈值']

kill switch：触发严重熔断时，一键停止开新仓、只减不增，必要时全部平仓并通知人工。


## 四、报警机制

风控异常必须让人知道，常见分级报警：
- **提示级**：日志、监控面板（如轻微偏离）；
- **警告级**：IM/邮件（如接近限额）；
- **紧急级**：短信、电话、自动熔断（如突破回撤、系统宕机）；
- 报警要有升级机制（无人响应则逐级上报），并支持人工接管。

## 五、监控：要盯哪些东西

| 监控类别 | 指标 |
|---|---|
| 系统状态 | 进程存活、CPU/内存、网络、延迟、行情是否正常 |
| 交易状态 | 订单/成交/拒单、未完成订单、对账差异 |
| 盈亏风险 | 实时净值、当日盈亏、回撤、敞口、风险因子暴露 |
| 数据质量 | 数据缺失、延迟、异常值 |

> 监控面板（如 Grafana）实时展示，配合阈值报警；关键指标异常要能第一时间发现。

## 六、日志与容灾

- **日志**：所有行情、信号、订单、成交、风控决策都要落日志，带时间戳，便于排查和审计；
- **容灾**：关键组件冗余（双机/备份），故障自动切换；
- **数据备份**：账户、持仓、订单定期备份，崩溃后能恢复到一致状态；
- **幂等**：重复发送同一指令不会造成重复下单。

In [3]:
# 事后报告：留档当日风控情况
report = rm.post_trade_report(1_000_000, {'A': 1, 'B': 2})
for k, v in report.items():
    print(k, '->', v)

nav -> 1000000
n_positions -> 2
alerts -> ['拒绝：B 权重 0.20 超上限', '熔断：现金比例 0.02 过低', '熔断：回撤 0.20 超阈值']


## 七、小结与面试要点

- 三道防线：事前拒错单、事中防失控（熔断/kill switch）、事后复盘对账。
- 限额 + 分级报警 + 监控面板 + 日志容灾，构成实盘安全网。
- 监控覆盖系统、交易、盈亏风险、数据质量四类。

**面试高频：**
- 实时风控在哪些环节做、各查什么？（答：事前单票/合规，事中回撤/敞口，事后对账）
- 什么是熔断 / kill switch？（答：触发阈值自动停止交易/只减不增/平仓并通知人工）
- 系统异常或宕机怎么处理？（答：报警、冗余切换、备份恢复、保证幂等）
- 报警怎么分级？（答：提示/警告/紧急，逐级升级）

## 动手练习
1. 为你的策略定义一套具体限额（单票/行业/现金/回撤）；
2. 模拟一次回撤超限，写出系统自动执行的动作序列；
3. 设计一张你自己的监控面板，列出要展示的指标。

---
下一节 **7.6 部署与运维：进程管理、定时任务、数据库、上线流程**。